# 🛣️ Kırşehir Yol Ağı — Sistematik Koordinat Toplama
# Street View Görüntü Sınıflandırma Projesi için
# OSM yol ağından her 50 metrede bir koordinat üretimi

In [33]:
# ============================================================
# Hücre 1: Gerekli kütüphanelerin kurulumu (Colab için)
# ============================================================
!pip install osmnx geopandas shapely pandas tqdm -q

In [34]:
# ============================================================
# Hücre 2: Kütüphanelerin import edilmesi
# ============================================================

import osmnx as ox          # OpenStreetMap yol ağı verisi indirme ve analiz
import geopandas as gpd      # Coğrafi veri çerçeveleri (GeoDataFrame)
import pandas as pd          # Tablo işlemleri ve CSV kaydetme
import numpy as np           # Aralık hesaplamaları (arange)
from shapely.geometry import Point  # Nokta geometrisi oluşturma
from tqdm import tqdm        # İlerleme çubuğu

print("✅ Tüm kütüphaneler başarıyla yüklendi.")
print(f"   OSMnx versiyonu : {ox.__version__}")
print(f"   GeoPandas vers. : {gpd.__version__}")

✅ Tüm kütüphaneler başarıyla yüklendi.
   OSMnx versiyonu : 2.1.0
   GeoPandas vers. : 1.1.3


In [35]:
# ============================================================
# Hücre 3: Kırşehir ŞEHİR MERKEZİ yol ağını OSM'den indirme
# ============================================================
# Bbox (bounding box) ile sadece şehir merkezini kapsıyoruz.
# Bu sayede köy yolları, şehirlerarası yollar ve bozkır dışarıda kalır.
#
# Kırşehir şehir merkezi bbox (yaklaşık):
#   Kuzey: 39.180, Güney: 39.125, Batı: 34.130, Doğu: 34.185

# Şehir merkezi sınırları (gerekirse Google Maps'ten ayarlayın)
NORTH, SOUTH = 39.180, 39.125
WEST, EAST   = 34.130, 34.185

print("⏳ Kırşehir şehir merkezi yol ağı indiriliyor (bbox ile)...")
print(f"   Bbox: N={NORTH}, S={SOUTH}, W={WEST}, E={EAST}")

# osmnx v2+ sürümlerinde bbox sırası: (west, south, east, north) şeklindedir
G = ox.graph_from_bbox(bbox=(WEST, SOUTH, EAST, NORTH), network_type="drive")

# Ağ hakkında özet bilgi
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()
print(f"✅ Yol ağı indirildi!")
print(f"   Düğüm (kavşak) sayısı : {num_nodes:,}")
print(f"   Kenar  (yol)    sayısı : {num_edges:,}")

⏳ Kırşehir şehir merkezi yol ağı indiriliyor (bbox ile)...
   Bbox: N=39.18, S=39.125, W=34.13, E=34.185
✅ Yol ağı indirildi!
   Düğüm (kavşak) sayısı : 2,920
   Kenar  (yol)    sayısı : 8,350


In [36]:
# ============================================================
# Hücre 4: Graf'ı UTM projeksiyonuna çevirme
# ============================================================
# Mesafeleri metre cinsinden doğru hesaplayabilmek için
# WGS84 (derece) yerine yerel UTM (metre) projeksiyonu kullanıyoruz.
# OSMnx bunu otomatik olarak en uygun UTM bölgesine çevirir.

print("⏳ Graf UTM projeksiyonuna çevriliyor...")
G_proj = ox.project_graph(G)

# Hangi CRS'ye çevrildiğini görelim
crs_info = ox.graph_to_gdfs(G_proj, edges=False).crs
print(f"✅ Projeksiyon tamamlandı!")
print(f"   Kullanılan CRS: {crs_info}")

⏳ Graf UTM projeksiyonuna çevriliyor...
✅ Projeksiyon tamamlandı!
   Kullanılan CRS: EPSG:32636


In [37]:
# ============================================================
# Hücre 5: Kenarları (yolları) GeoDataFrame olarak çıkarma
# ============================================================
# Projekte edilmiş graftan sadece kenarları (edges) alıyoruz.
# Her kenar, bir yol segmentini temsil eden bir LineString geometrisi taşır.

print("⏳ Kenarlar GeoDataFrame'e dönüştürülüyor...")
edges_gdf = ox.graph_to_gdfs(G_proj, nodes=False, edges=True)
print(f"   Ham kenar sayısı: {len(edges_gdf):,}")

# ─── FİLTRE 1: İsimsiz yolları çıkar ─────────────────────
# İsimsiz yollar genelde tali yollar, boş araziye giden patikalar vs.
# ML eğitiminde kullanılmayacak, fotoğraf çekmeye gerek yok.

before = len(edges_gdf)
edges_gdf[["geometry", "length"]].head(3)

edges_gdf = edges_gdf[edges_gdf["name"].notna()].copy()
print(f"\n   İlk 3 yol segmenti:")

print(f"   ❌ İsimsiz yollar çıkarıldı: {before:,} → {len(edges_gdf):,} ({before - len(edges_gdf):,} silindi)")
print(f"   CRS: {edges_gdf.crs}")

print(f"\n✅ Filtrelenmiş toplam: {len(edges_gdf):,} yol segmenti")

# ─── FİLTRE 2: Otoyol & şehirlerarası yolları çıkar ──────

EXCLUDE_HIGHWAY_TYPES = {"motorway", "trunk", "motorway_link", "trunk_link"}
before = len(edges_gdf)

def should_exclude_highway(highway_val):
    """highway sütunu list veya str olabilir, her ikisini de kontrol et."""
    if isinstance(highway_val, list):
        return any(h in EXCLUDE_HIGHWAY_TYPES for h in highway_val)
    return highway_val in EXCLUDE_HIGHWAY_TYPES

edges_gdf = edges_gdf[~edges_gdf["highway"].apply(should_exclude_highway)].copy()
print(f"   ❌ Otoyol/trunk yollar çıkarıldı: {before:,} → {len(edges_gdf):,} ({before - len(edges_gdf):,} silindi)")

⏳ Kenarlar GeoDataFrame'e dönüştürülüyor...
   Ham kenar sayısı: 8,350

   İlk 3 yol segmenti:
   ❌ İsimsiz yollar çıkarıldı: 8,350 → 3,987 (4,363 silindi)
   CRS: EPSG:32636

✅ Filtrelenmiş toplam: 3,987 yol segmenti
   ❌ Otoyol/trunk yollar çıkarıldı: 3,987 → 3,934 (53 silindi)


In [38]:
# ============================================================
# Hücre 6: Yol üzerinde Her 50 metrede bir nokta üretimi
# ============================================================
# Her yol segmenti (LineString) üzerinde sabit aralıklarla
# shapely.interpolate() kullanarak sistematik noktalar üretiyoruz.
# Bu sayede bozkır/boş arazi yerine sadece gerçek yol üstü
# koordinatları elde ediyoruz.

INTERVAL = 50  # metre — noktalar arası mesafe (değiştirilebilir)

print(f"⏳ Her {INTERVAL} metrede bir nokta üretiliyor...")

points = []  # Üretilen tüm Point geometrilerini tutacak liste

for _, row in tqdm(edges_gdf.iterrows(), total=len(edges_gdf),
                   desc="Yol segmentleri işleniyor"):
    line = row.geometry           # LineString geometrisi
    line_length = line.length     # Metre cinsinden yol uzunluğu

    # 0'dan yol sonuna kadar INTERVAL aralıkla mesafe değerleri
    distances = np.arange(0, line_length, INTERVAL)

    # Her mesafe için yol üzerindeki noktayı hesapla
    for dist in distances:
        point = line.interpolate(dist)  # Shapely interpolation
        points.append(point)

    # Son noktayı da ekle (yolun tam ucundaki nokta)
    points.append(line.interpolate(line_length))

print(f"✅ Toplam {len(points):,} nokta üretildi.")

# Noktaları GeoDataFrame'e çevir (UTM CRS ile)
points_gdf = gpd.GeoDataFrame(
    geometry=points,
    crs=edges_gdf.crs  # Hâlâ UTM projeksiyonunda
)
print(f"   GeoDataFrame boyutu: {points_gdf.shape}")

⏳ Her 50 metrede bir nokta üretiliyor...


Yol segmentleri işleniyor: 100%|██████████| 3934/3934 [00:00<00:00, 5308.79it/s]


✅ Toplam 12,957 nokta üretildi.
   GeoDataFrame boyutu: (12957, 1)


In [39]:
# ============================================================
# Hücre 7: UTM → WGS84 (Enlem/Boylam) dönüşümü
# ============================================================
# Street View API'leri ve harita uygulamaları standart enlem/boylam
# (EPSG:4326) koordinatları bekler. UTM'den geri çeviriyoruz.

print("⏳ Koordinatlar WGS84 (EPSG:4326) formatına çevriliyor...")

points_wgs84 = points_gdf.to_crs(epsg=4326)

# Enlem (latitude = y) ve Boylam (longitude = x) sütunlarını çıkar
points_wgs84["latitude"]  = points_wgs84.geometry.y
points_wgs84["longitude"] = points_wgs84.geometry.x

print(f"✅ Dönüşüm tamamlandı!")
print(f"   Örnek koordinatlar:")
points_wgs84[["latitude", "longitude"]].head()

⏳ Koordinatlar WGS84 (EPSG:4326) formatına çevriliyor...
✅ Dönüşüm tamamlandı!
   Örnek koordinatlar:


,latitude,longitude
0,39.167404,34.151911
1,39.167233,34.151809
2,39.167233,34.151809
3,39.167404,34.151911
4,39.167233,34.151809


In [40]:
# ============================================================
# Hücre 8: Koordinatları CSV olarak kaydetme
# ============================================================
# Tekrarlı noktaları temizleyip sadece lat/lon olarak kaydediyoruz.

OUTPUT_FILE = "kirsehir_systematic_coords.csv"

# Sadece enlem/boylam sütunlarını al
df = points_wgs84[["latitude", "longitude"]].copy()

# Aynı noktada birden fazla kayıt varsa temizle (kenar kesişimleri vb.)
before_dedup = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after_dedup = len(df)

print(f"🔄 Tekrarlı noktalar temizlendi: {before_dedup:,} → {after_dedup:,} "
      f"({before_dedup - after_dedup:,} tekrar silindi)")

# CSV olarak kaydet
df.to_csv(OUTPUT_FILE, index=False)

print(f"\n✅ Dosya kaydedildi: '{OUTPUT_FILE}'")
print(f"   Toplam koordinat sayısı: {len(df):,}")
print(f"\n   İlk 10 satır:")
df.head(10)

🔄 Tekrarlı noktalar temizlendi: 12,957 → 6,800 (6,157 tekrar silindi)

✅ Dosya kaydedildi: 'kirsehir_systematic_coords.csv'
   Toplam koordinat sayısı: 6,800

   İlk 10 satır:


,latitude,longitude
0,39.167404,34.151911
1,39.167233,34.151809
2,39.166833,34.152038
3,39.167245,34.151686
4,39.149155,34.166376
5,39.149248,34.166042
6,39.148895,34.165903
7,39.148636,34.165431
8,39.148376,34.164958
9,39.148126,34.164477


In [41]:
# ============================================================
# Hücre 9: Doğrulama — Koordinatların Kırşehir sınırları içinde
#           olduğunu ve dosyanın düzgün oluştuğunu kontrol et
# ============================================================

import os

# 1) Dosya varlığı kontrolü
assert os.path.exists(OUTPUT_FILE), f"❌ {OUTPUT_FILE} bulunamadı!"

# 2) Dosyayı tekrar oku ve kontrol et
df_check = pd.read_csv(OUTPUT_FILE)
print(f"📄 '{OUTPUT_FILE}' başarıyla okundu: {len(df_check):,} satır")

# 3) Kırşehir merkez koordinatları yaklaşık: 39.14°N, 34.16°E
# Tüm noktaların makul bir sınır içinde olup olmadığını kontrol et
lat_min, lat_max = df_check["latitude"].min(),  df_check["latitude"].max()
lon_min, lon_max = df_check["longitude"].min(), df_check["longitude"].max()

print(f"\n   Enlem  aralığı: {lat_min:.4f}° — {lat_max:.4f}°")
print(f"   Boylam aralığı: {lon_min:.4f}° — {lon_max:.4f}°")

# Basit sınır kontrolü (Kırşehir il sınırları civarı)
assert 38.5 < lat_min and lat_max < 40.0, "⚠️ Enlem değerleri beklenen aralıkta değil!"
assert 33.0 < lon_min and lon_max < 35.5, "⚠️ Boylam değerleri beklenen aralıkta değil!"

print(f"\n✅ Tüm doğrulamalar başarılı!")
print(f"   {len(df_check):,} koordinat Kırşehir sınırları içinde.")
print(f"   Veri, Street View görüntü toplama için hazır. 🎯")

📄 'kirsehir_systematic_coords.csv' başarıyla okundu: 6,800 satır

   Enlem  aralığı: 39.1250° — 39.1799°
   Boylam aralığı: 34.1300° — 34.1848°

✅ Tüm doğrulamalar başarılı!
   6,800 koordinat Kırşehir sınırları içinde.
   Veri, Street View görüntü toplama için hazır. 🎯


In [42]:
# ============================================================
# Hücre 10: CSV dosyasını indirme
# ============================================================
# SEÇENEK A — Doğrudan bilgisayara indir (tarayıcı üzerinden)
# ============================================================
from google.colab import files
files.download(OUTPUT_FILE)
print("⬇️  İndirme başlatıldı — tarayıcının indirme klasörüne bakın.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  İndirme başlatıldı — tarayıcının indirme klasörüne bakın.


In [43]:
# ============================================================
# SEÇENEK B — Google Drive'a kaydet (kalıcı depolama)
# ============================================================
# Bu hücreyi çalıştırınca Drive bağlantısı için izin istenir.
# Dosya "My Drive/kirsehir_systematic_coords.csv" olarak kaydedilir.

import shutil
from google.colab import drive

# Drive'ı bağla (/content/drive altında mount edilir)
drive.mount("/content/drive")

# Hedef klasörü ve yolu belirle
DRIVE_DEST = "/content/drive/My Drive/kirsehir_systematic_coords.csv"

# Dosyayı Drive'a kopyala
shutil.copy(OUTPUT_FILE, DRIVE_DEST)
print(f"✅ Dosya Google Drive'a kopyalandı: {DRIVE_DEST}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Dosya Google Drive'a kopyalandı: /content/drive/My Drive/kirsehir_systematic_coords.csv


---
# 📸 Bölüm 2: Street View Veri Toplama
## Google Geocoding + Street View Static API ile mahalle bazlı fotoğraf toplama

### 🔑 API Anahtarı Nasıl Oluşturulur?
1. [Google Cloud Console](https://console.cloud.google.com/) → **APIs & Services** → **Credentials**
2. **"+ CREATE CREDENTIALS"** → **"API key"** seçin
3. **APIs & Services** → **Library** sayfasından şu iki API'yi etkinleştirin:
   - **Geocoding API**
   - **Street View Static API**
4. **Billing** → Bir faturalandırma hesabı bağlayın (Yeni hesaplara **$300 ücretsiz kredi** verilir)
5. Oluşan API anahtarını aşağıdaki hücrede girmeniz istenecek

> ⚠️ API anahtarınızı asla notebook'a doğrudan yazmayın — `getpass` ile güvenli giriş kullanılır.

In [44]:
# ============================================================
# Hücre 12: Konfigürasyon & Import
# ============================================================

import requests          # HTTP istekleri (API çağrıları)
import os                # Dosya/klasör işlemleri
import time              # Rate limiting için bekleme
import math              # Haversine mesafe hesabı
import json              # JSON parse
import pandas as pd      # CSV okuma/yazma
from tqdm import tqdm    # İlerleme çubuğu
from getpass import getpass  # API key güvenli giriş

# ─── API Anahtarı (güvenli giriş) ─────────────────────────
API_KEY = getpass("🔑 Google API anahtarınızı yapıştırın: ")
assert len(API_KEY) > 10, "❌ Geçersiz API anahtarı!"
print(f"✅ API anahtarı alındı (ilk 8 karakter: {API_KEY[:8]}...)")

# ─── Sabitler ──────────────────────────────────────────────
IMG_SIZE        = "640x640"           # Fotoğraf çözünürlüğü
HEADINGS        = [0, 90, 180, 270]   # Kuzey, Doğu, Güney, Batı
HEADING_NAMES   = {0: "N", 90: "E", 180: "S", 270: "W"}
FOV             = 90                  # Field of View (derece)
CHECKPOINT_EVERY = 500                # Her 500 noktada Drive'a kaydet
MAX_SNAP_DISTANCE = 50                # metre — panorama, istenen noktadan bu kadar uzaksa reddet
RATE_LIMIT_DELAY  = 0.05

# ─── Drive Klasör Yapısı ──────────────────────────────────
DRIVE_ROOT     = "/content/drive/My Drive/Kirsehir_ML_Data"
PROGRESS_FILE  = os.path.join(DRIVE_ROOT, "progress_log.csv")
CSV_INPUT      = "kirsehir_systematic_coords.csv"

# ─── Geocoding Cache Ayarları ─────────────────────────────
# Koordinatları ~11m (0.001 derece) hassasiyetle yuvarla
# Aynı grid hücresindeki noktalar için tekrar sorgu yapmaz
GRID_PRECISION = 4  # ondalık basamak (3 → ~11m grid)

# ─── API Endpoint'leri ────────────────────────────────────
GEOCODING_URL   = "https://maps.googleapis.com/maps/api/geocode/json"
SV_METADATA_URL = "https://maps.googleapis.com/maps/api/streetview/metadata"
SV_IMAGE_URL    = "https://maps.googleapis.com/maps/api/streetview"

print(f"\n📋 Konfigürasyon:")
print(f"   Fotoğraf boyutu  : {IMG_SIZE}")
print(f"   Yön sayısı       : {len(HEADINGS)} (K-D-G-B)")
print(f"   Checkpoint aralığı: Her {CHECKPOINT_EVERY} nokta")
print(f"   Drive klasörü    : {DRIVE_ROOT}")

print(f"   Grid hassasiyeti : ~{111 * 10**(-GRID_PRECISION + 3):.0f}m")
print(f"   Grid hassasiyeti : ~{111 * 10**(-GRID_PRECISION + 3):.0f}m")

✅ API anahtarı alındı (ilk 8 karakter: AIzaSyCh...)

📋 Konfigürasyon:
   Fotoğraf boyutu  : 640x640
   Yön sayısı       : 4 (K-D-G-B)
   Checkpoint aralığı: Her 500 nokta
   Drive klasörü    : /content/drive/My Drive/Kirsehir_ML_Data
   Grid hassasiyeti : ~11m
   Grid hassasiyeti : ~11m


In [45]:
# ============================================================
# Hücre 13: Drive Bağlantısı & Ana Klasör Oluşturma
# ============================================================

from google.colab import drive

# Drive'ı bağla (daha önce bağlandıysa tekrar istemez)
drive.mount("/content/drive")

# Ana proje klasörünü oluştur
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f"✅ Drive bağlandı & ana klasör hazır: {DRIVE_ROOT}")

# CSV dosyasını oku
df_coords = pd.read_csv(CSV_INPUT)
total_points = len(df_coords)
print(f"📄 CSV yüklendi: {total_points:,} koordinat")

# ─── Sıfırdan başlama (önceki verilerde sorun vardı) ──────
# ÖNCEKİ PROGRESS LOG'U SİL → temiz yeniden başlangıç
# Filtreler değiştiği için eski log artık geçersiz.
if os.path.exists(PROGRESS_FILE):
    os.remove(PROGRESS_FILE)
    print("🗑️  Eski progress_log.csv silindi (filtreler değişti, sıfırdan başlanıyor)")

completed_indices = set()
print("🆕 Yeni oturum — temiz başlangıç.")

print(f"\n   Tahmini fotoğraf sayısı: ~{(total_points - len(completed_indices)) * 4:,} adet")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive bağlandı & ana klasör hazır: /content/drive/My Drive/Kirsehir_ML_Data
📄 CSV yüklendi: 6,800 koordinat
🆕 Yeni oturum — temiz başlangıç.

   Tahmini fotoğraf sayısı: ~27,200 adet


In [46]:
# ============================================================
# Hücre 14: Geocoding Cache Sistemi (Bütçe Optimizasyonu)
# ============================================================
# Aynı ~11m grid hücresindeki tüm noktalar aynı mahallede kabul edilir.
# Böylece 5000 Geocoding çağrısı yerine ~300-500 çağrı yapılır (%90+ tasarruf).

geocoding_cache = {}  # {(grid_lat, grid_lon): mahalle_adi}
geocoding_api_calls = 0  # Toplam API çağrısı sayacı


def get_grid_key(lat, lon):
    """Koordinatı grid hücresine yuvarla (cache key olarak kullan)."""
    return (round(lat, GRID_PRECISION), round(lon, GRID_PRECISION))


def geocode_to_neighborhood(lat, lon):
    """
    Google Geocoding API ile koordinattan mahalle adını bul.
    Önce cache'e bakar; yoksa API'ye sorar ve cache'e ekler.

    Aranan bileşen sırası:
      1. neighborhood       (mahalle)
      2. sublocality_level_1 (alt bölge)
      3. administrative_area_level_4
      4. route               (sokak/cadde adı)
      5. "Bilinmeyen"        (hiçbiri bulunamazsa)
    """
    global geocoding_api_calls

    # ─── Cache kontrolü ───────────────────────────────────
    grid_key = get_grid_key(lat, lon)
    if grid_key in geocoding_cache:
        return geocoding_cache[grid_key]

    # ─── API çağrısı ──────────────────────────────────────
    params = {
        "latlng": f"{lat},{lon}",
        "key": API_KEY,
        "language": "tr",          # Türkçe sonuç
        "result_type": "neighborhood|sublocality|administrative_area_level_4|route"
    }

    try:
        resp = requests.get(GEOCODING_URL, params=params, timeout=10)
        data = resp.json()
        geocoding_api_calls += 1

        if data["status"] == "OK" and data["results"]:
            # Tüm address_components içinde mahalle ara
            # Öncelik sırası ile dene
            priority = [
                "neighborhood",
                "sublocality_level_1",
                "sublocality",
                "administrative_area_level_4",
                "route"
            ]

            for result in data["results"]:
                for component in result.get("address_components", []):
                    types = component.get("types", [])
                    for p in priority:
                        if p in types:
                            name = component["long_name"]
                            # Türkçe karakter sorunlarını önle (dosya adı için)
                            safe_name = name.replace("/", "-").replace("\\", "-").strip()
                            geocoding_cache[grid_key] = safe_name
                            return safe_name

        # Hiçbir bileşen bulunamadı
        geocoding_cache[grid_key] = "Bilinmeyen"
        return "Bilinmeyen"

    except Exception as e:
        print(f"   ⚠️ Geocoding hatası ({lat:.4f}, {lon:.4f}): {e}")
        geocoding_cache[grid_key] = "Bilinmeyen"
        return "Bilinmeyen"


# Test et (ilk koordinat ile)
test_lat = df_coords.iloc[0]["latitude"]
test_lon = df_coords.iloc[0]["longitude"]
test_mahalle = geocode_to_neighborhood(test_lat, test_lon)
print(f"🧪 Test: ({test_lat:.4f}, {test_lon:.4f}) → Mahalle: \"{test_mahalle}\"")
print(f"   Cache boyutu: {len(geocoding_cache)}, API çağrısı: {geocoding_api_calls}")

🧪 Test: (39.1674, 34.1519) → Mahalle: "Atatürk Caddesi"
   Cache boyutu: 1, API çağrısı: 1


In [47]:
# ============================================================
# Hücre 15: Street View Yardımcı Fonksiyonları
# ============================================================
# Metadata kontrolü (ücretsiz) + Fotoğraf indirme fonksiyonları


def check_sv_coverage(lat, lon):
    """
    Street View metadata endpoint'ine sorgu at (ÜCRETSİZ).
    Kapsam varsa True, yoksa False döndürür.
    source=outdoor → sadece dış mekan panoramaları kontrol edilir.
    Bu sayede indoor fotoğraflar ve kapsamı olmayan noktalar elenir.
    """
    params = {
        "location": f"{lat},{lon}",
        "source": "outdoor",   # ← Indoor panoramaları hariç tut
        "key": API_KEY
    }
    try:
        resp = requests.get(SV_METADATA_URL, params=params, timeout=10)
        data = resp.json()

        if data.get("status") != "OK":
            return False, None  # Kapsam yok

        # ─── Snapping mesafe kontrolü ──────────────────────
        # API en yakın panoramayı döndürür; ancak bu bazen çok uzak olabilir.
        # Panoramanın gerçek konumunu alıp istenen noktaya olan mesafeyi ölç.
        pano_lat = data["location"]["lat"]
        pano_lon = data["location"]["lng"]
        pano_id  = data["pano_id"]

        # Haversine formülü ile metre cinsinden mesafe
        R = 6371000
        phi1, phi2 = math.radians(lat), math.radians(pano_lat)
        dphi    = math.radians(pano_lat - lat)
        dlambda = math.radians(pano_lon - lon)
        a    = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
        dist = 2 * R * math.asin(math.sqrt(a))

        if dist > MAX_SNAP_DISTANCE:
            # Panorama çok uzakta → snapping olmuş, reddet
            return False, None

        return True, pano_id  # Kapsam var + panorama ID'si

    except Exception:
        return False, None  # Hata durumunda atla


def download_sv_images(lat, lon, save_dir, pano_id):
    """
    Bir koordinat için 4 yönlü (K-D-G-B) Street View fotoğrafı indir.
    pano_id kullanarak EXACT panoramayı çeker → snapping olmaz.
    Fotoğrafları doğrudan Drive klasörüne kaydeder.

    Dosya adı formatı: {lat}_{lon}_h{heading}_{yön_harfi}.jpg

    Returns:
        int: Başarıyla indirilen fotoğraf sayısı
    """
    downloaded = 0

    for heading in HEADINGS:
        params = {
            "pano": pano_id,        # ← Koordinat yerine Pano ID → kesin panorama, snap yok
            "size": IMG_SIZE,
            "heading": heading,
            "fov": FOV,
            "pitch": 0,
            "key": API_KEY
        }

        try:
            resp = requests.get(SV_IMAGE_URL, params=params, timeout=15)

            if resp.status_code == 200 and resp.headers.get("content-type", "").startswith("image"):
                # Dosya adı: lat_lon_hHeading_Yön.jpg
                direction = HEADING_NAMES[heading]
                filename = f"{lat:.6f}_{lon:.6f}_h{heading}_{direction}.jpg"
                filepath = os.path.join(save_dir, filename)


                with open(filepath, "wb") as f:
                    f.write(resp.content)
                downloaded += 1

            # Rate limiting
            time.sleep(RATE_LIMIT_DELAY)

        except Exception as e:
            print(f"   ⚠️ İndirme hatası ({lat:.4f}, {lon:.4f}, h={heading}): {e}")
            continue

    return downloaded


def save_progress(log_records):
    """Progress log'u Drive'a kaydet (checkpoint)."""
    df_log = pd.DataFrame(log_records)
    df_log.to_csv(PROGRESS_FILE, index=False)


In [48]:
# ============================================================
# Hücre 16: 🚀 ANA İNDİRME DÖNGÜSÜ
# ============================================================
# CSV'deki her koordinat için:
#   1) Geocoding cache ile mahalle adını bul
#   2) Drive'da mahalle klasörü aç (yoksa)
#   3) Street View metadata kontrolü (ücretsiz)
#   4) Kapsam varsa → 4 yönlü fotoğraf indir
#   5) Her 500 noktada checkpoint kaydet
#
# Colab çökerse → Hücre 13'ten itibaren tekrar çalıştırın,
# progress_log.csv sayesinde kaldığı yerden devam eder.

# ─── Önceki oturumdan log kayıtlarını yükle (varsa) ──────
if os.path.exists(PROGRESS_FILE):
    existing_log = pd.read_csv(PROGRESS_FILE).to_dict("records")
else:
    existing_log = []

log_records = list(existing_log)  # Progress log kayıtları

# ─── Sayaçlar ─────────────────────────────────────────────
stats = {
    "processed": len(completed_indices),
    "downloaded_photos": 0,
    "no_coverage": 0,
    "errors": 0,
    "geocoding_cache_hits": 0
}

print("=" * 60)
print("🚀 STREET VIEW VERİ TOPLAMA BAŞLIYOR")
print("=" * 60)
print(f"   Toplam nokta    : {total_points:,}")
print(f"   Zaten işlenmiş  : {len(completed_indices):,}")
print(f"   İşlenecek       : {total_points - len(completed_indices):,}")
print(f"   Checkpoint      : Her {CHECKPOINT_EVERY} noktada")
print("=" * 60)

# ─── Ana döngü ────────────────────────────────────────────
progress_bar = tqdm(total=total_points, initial=len(completed_indices),
                    desc="Toplam ilerleme", unit="nokta")

for idx, row in df_coords.iterrows():
    # Zaten işlenmişse atla (resume desteği)
    if idx in completed_indices:
        continue

    lat = row["latitude"]
    lon = row["longitude"]

    try:
        # ─── 1) Mahalle adını bul (cache ile) ─────────────
        grid_key = get_grid_key(lat, lon)
        cache_hit = grid_key in geocoding_cache
        mahalle = geocode_to_neighborhood(lat, lon)

        if cache_hit:
            stats["geocoding_cache_hits"] += 1

        # ─── 2) Mahalle klasörünü oluştur ─────────────────
        mahalle_dir = os.path.join(DRIVE_ROOT, mahalle)
        os.makedirs(mahalle_dir, exist_ok=True)

        # ─── 3) Street View kapsam + snapping mesafe kontrolü (ücretsiz) ─
        # check_sv_coverage: panorama 50m'den uzaksa (snap) False döner
        has_coverage, pano_id = check_sv_coverage(lat, lon)

        if has_coverage:
            # ─── 4) 4 yönlü fotoğraf indir (pano ID ile, snap yok) ──────
            num_photos = download_sv_images(lat, lon, mahalle_dir, pano_id)
            stats["downloaded_photos"] += num_photos

            log_records.append({
                "index": idx,
                "latitude": lat,
                "longitude": lon,
                "mahalle": mahalle,
                "status": "success",
                "photos": num_photos
            })
        else:
            # Kapsam yok — atla
            stats["no_coverage"] += 1
            log_records.append({
                "index": idx,
                "latitude": lat,
                "longitude": lon,
                "mahalle": mahalle,
                "status": "no_coverage",
                "photos": 0
            })

    except Exception as e:
        stats["errors"] += 1
        log_records.append({
            "index": idx,
            "latitude": lat,
            "longitude": lon,
            "mahalle": "Hata",
            "status": f"error: {str(e)[:50]}",
            "photos": 0
        })

    stats["processed"] += 1
    completed_indices.add(idx)
    progress_bar.update(1)

    # ─── 5) Checkpoint: her 500 noktada kaydet ────────────
    points_since_start = stats["processed"] - len(existing_log)
    if points_since_start > 0 and points_since_start % CHECKPOINT_EVERY == 0:
        save_progress(log_records)
        print(f"\n   💾 CHECKPOINT @ {stats['processed']:,} nokta | "
              f"📸 {stats['downloaded_photos']:,} fotoğraf | "
              f"🚫 {stats['no_coverage']:,} kapsam dışı | "
              f"❌ {stats['errors']} hata | "
              f"🗂️ Cache: {len(geocoding_cache)} grid, "
              f"Geocoding API: {geocoding_api_calls} çağrı")

progress_bar.close()

# ─── Son kayıt ────────────────────────────────────────────
save_progress(log_records)

print("\n" + "=" * 60)
print("✅ VERİ TOPLAMA TAMAMLANDI!")
print("=" * 60)
print(f"   📍 İşlenen nokta        : {stats['processed']:,}")
print(f"   📸 İndirilen fotoğraf   : {stats['downloaded_photos']:,}")
print(f"   🚫 Kapsam dışı nokta    : {stats['no_coverage']:,}")
print(f"   ❌ Hata                  : {stats['errors']}")
print(f"   🗂️  Geocoding API çağrısı: {geocoding_api_calls}")
print(f"   💰 Cache hit sayısı     : {stats['geocoding_cache_hits']:,}")
print(f"   📁 Veriler: {DRIVE_ROOT}")
print("=" * 60)

🚀 STREET VIEW VERİ TOPLAMA BAŞLIYOR
   Toplam nokta    : 6,800
   Zaten işlenmiş  : 0
   İşlenecek       : 6,800
   Checkpoint      : Her 500 noktada


Toplam ilerleme:   7%|▋         | 500/6800 [05:44<42:57,  2.44nokta/s]  


   💾 CHECKPOINT @ 500 nokta | 📸 1,976 fotoğraf | 🚫 6 kapsam dışı | ❌ 0 hata | 🗂️ Cache: 468 grid, Geocoding API: 468 çağrı


Toplam ilerleme:  15%|█▍        | 1000/6800 [10:33<1:26:31,  1.12nokta/s]


   💾 CHECKPOINT @ 1,000 nokta | 📸 3,892 fotoğraf | 🚫 27 kapsam dışı | ❌ 0 hata | 🗂️ Cache: 937 grid, Geocoding API: 937 çağrı


...


   💾 CHECKPOINT @ 6,000 nokta | 📸 23,292 fotoğraf | 🚫 177 kapsam dışı | ❌ 0 hata | 🗂️ Cache: 5455 grid, Geocoding API: 5455 çağrı


Toplam ilerleme:  96%|█████████▌| 6500/6800 [1:21:05<04:40,  1.07nokta/s]


   💾 CHECKPOINT @ 6,500 nokta | 📸 25,184 fotoğraf | 🚫 204 kapsam dışı | ❌ 0 hata | 🗂️ Cache: 5890 grid, Geocoding API: 5890 çağrı


Toplam ilerleme: 100%|██████████| 6800/6800 [1:24:33<00:00,  1.34nokta/s]


✅ VERİ TOPLAMA TAMAMLANDI!
   📍 İşlenen nokta        : 6,800
   📸 İndirilen fotoğraf   : 26,252
   🚫 Kapsam dışı nokta    : 237
   ❌ Hata                  : 0
   🗂️  Geocoding API çağrısı: 6148
   💰 Cache hit sayısı     : 653
   📁 Veriler: /content/drive/My Drive/Kirsehir_ML_Data


In [49]:
# ============================================================
# Hücre 17: 📊 Özet İstatistikler & Mahalle Dağılımı
# ============================================================

# Progress log'u oku
df_log = pd.read_csv(PROGRESS_FILE)

print("=" * 60)
print("📊 VERİ TOPLAMA İSTATİSTİKLERİ")
print("=" * 60)

# Genel durum dağılımı
status_counts = df_log["status"].value_counts()
print(f"\n🔹 Durum Dağılımı:")
for status, count in status_counts.items():
    print(f"   {status:20s}: {count:,}")

# Başarılı noktaların mahalle dağılımı
df_success = df_log[df_log["status"] == "success"]
total_photos = df_success["photos"].sum()

print(f"\n🔹 Toplam indirilen fotoğraf: {int(total_photos):,}")
print(f"🔹 Kapsam olan nokta sayısı : {len(df_success):,}")

# Mahalle bazında dağılım
if len(df_success) > 0:
    mahalle_stats = (
        df_success
        .groupby("mahalle")
        .agg(nokta_sayisi=("index", "count"), foto_sayisi=("photos", "sum"))
        .sort_values("foto_sayisi", ascending=False)
        .reset_index()
    )
    mahalle_stats["foto_sayisi"] = mahalle_stats["foto_sayisi"].astype(int)

    print(f"\n🔹 Mahalle Bazında Dağılım ({len(mahalle_stats)} mahalle):")
    print("-" * 50)
    print(f"   {'Mahalle':<25s} {'Nokta':>8s} {'Fotoğraf':>10s}")
    print("-" * 50)
    for _, r in mahalle_stats.iterrows():
        print(f"   {r['mahalle']:<25s} {r['nokta_sayisi']:>8,} {r['foto_sayisi']:>10,}")
    print("-" * 50)

# ─── Tahmini maliyet ──────────────────────────────────────
geocoding_cost = geocoding_api_calls * 0.005      # $5/1000
sv_image_cost  = int(total_photos) * 0.007         # $7/1000
total_cost     = geocoding_cost + sv_image_cost

print(f"\n💰 Tahmini API Maliyeti:")
print(f"   Geocoding  : {geocoding_api_calls:,} çağrı × $0.005 = ${geocoding_cost:.2f}")
print(f"   Street View: {int(total_photos):,} foto  × $0.007 = ${sv_image_cost:.2f}")
print(f"   ─────────────────────────────────────")
print(f"   TOPLAM     : ${total_cost:.2f}")
print(f"\n📁 Tüm veriler: {DRIVE_ROOT}")
print("=" * 60)

📊 VERİ TOPLAMA İSTATİSTİKLERİ

🔹 Durum Dağılımı:
   success             : 6,563
   no_coverage         : 237

🔹 Toplam indirilen fotoğraf: 26,252
🔹 Kapsam olan nokta sayısı : 6,563

🔹 Mahalle Bazında Dağılım (654 mahalle):
--------------------------------------------------
   Mahalle                      Nokta   Fotoğraf
--------------------------------------------------
   Obruk Caddesi                  113        452
   Boztepe Caddesi                103        412
   653. Sokak                      90        360
   Atatürk Bulvarı                 88        352
   Atatürk Caddesi                 67        268
   Kuyubaşı Caddesi                67        268
   Şehit Sahir Kurutluoğlu Caddesi       65        260
   Terme Caddesi                   57        228
   Doktor Sadık Ahmet Bulvarı       57        228
   Mustafa Kemal Paşa Caddesi       56        224
   Gazi Osman Paşa Bulvarı         51        204
   Şehit Mustafa Atasoy Caddesi       50        200
   Aşık Paşa Bulvarı       